# BraTS 2020 – Swin-UNETR (Hierarchical Swin Transformer Backbone)
Swin-UNETR replaces the global ViT with a **Swin Transformer** that uses shifted-window self-attention, yielding a hierarchical feature pyramid for richer multi-scale representations.

In [1]:
!pip install nibabel tqdm --quiet

In [2]:
import os
import math
import time
import random
import itertools
import numpy as np
import nibabel as nib
from glob import glob
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [3]:
SEED        = 42
IMG_SIZE    = 128
EPOCHS      = 10
BATCH_SIZE  = 2       # Swin-UNETR is memory-heavy; reduce if OOM
LR          = 1e-4
NUM_WORKERS = 4
PATIENCE    = 6

# Swin-UNETR hyper-params
PATCH_SIZE    = 2     # token patch size (2x2x2 voxels)
WINDOW_SIZE   = 7     # attention window (7x7x7 tokens)
EMBED_DIM     = 48    # initial embed dim; doubles at each stage
DEPTHS        = (2, 2, 2, 2)   # layers per stage
NUM_HEADS     = (3, 6, 12, 24) # heads per stage

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}:", torch.cuda.get_device_name(i))

CUDA: True
GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [4]:
DATA_ROOT  = "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation"
TRAIN_PATH = os.path.join(DATA_ROOT, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData")

all_cases = sorted(glob(os.path.join(TRAIN_PATH, "BraTS20_Training_*")))
all_cases = [c for c in all_cases if os.path.exists(f"{c}/{os.path.basename(c)}_seg.nii")]

print("Total cases:", len(all_cases))

Total cases: 368


In [5]:
def load_and_preprocess(path):
    cid = os.path.basename(path)

    def load_mod(mod):
        vol  = nib.load(f"{path}/{cid}_{mod}.nii").get_fdata().astype(np.float32)
        mask = vol != 0
        if mask.sum() > 0:
            vol[mask] = (vol[mask] - vol[mask].mean()) / (vol[mask].std() + 1e-8)
        t = torch.from_numpy(vol).unsqueeze(0).unsqueeze(0)
        t = F.interpolate(t, size=(IMG_SIZE, IMG_SIZE, IMG_SIZE),
                          mode="trilinear", align_corners=False)
        return t.squeeze().numpy().astype(np.float16)

    flair = load_mod("flair")
    t1    = load_mod("t1")
    t1ce  = load_mod("t1ce")
    t2    = load_mod("t2")

    seg = nib.load(f"{path}/{cid}_seg.nii").get_fdata().astype(np.float32)
    t   = torch.from_numpy(seg).unsqueeze(0).unsqueeze(0)
    t   = F.interpolate(t, size=(IMG_SIZE, IMG_SIZE, IMG_SIZE), mode="nearest")
    seg = t.squeeze().numpy()

    wt = (seg > 0).astype(np.float16)
    tc = ((seg == 1) | (seg == 4)).astype(np.float16)
    et = (seg == 4).astype(np.float16)

    image = np.stack([flair, t1, t1ce, t2], axis=0)
    label = np.stack([wt, tc, et], axis=0)
    return image, label


print("Preprocessing and caching all volumes to RAM...")
cache_start = time.time()

image_cache = {}
label_cache = {}

for path in tqdm(all_cases):
    img, lbl          = load_and_preprocess(path)
    image_cache[path] = img
    label_cache[path] = lbl

cache_time = time.time() - cache_start
print(f"Cache built in {cache_time:.1f}s  ({cache_time/60:.1f}min)")

total_mb = sum(v.nbytes for v in image_cache.values()) + sum(v.nbytes for v in label_cache.values())
print(f"Total RAM used: {total_mb / 1e9:.2f} GB")

Preprocessing and caching all volumes to RAM...


  0%|          | 0/368 [00:00<?, ?it/s]

Cache built in 715.9s  (11.9min)
Total RAM used: 10.80 GB


In [6]:
class BraTSDataset(Dataset):
    def __init__(self, cases, augment=False):
        self.cases   = cases
        self.augment = augment

    def __len__(self):
        return len(self.cases)

    def __getitem__(self, idx):
        path  = self.cases[idx]
        image = torch.from_numpy(image_cache[path].astype(np.float32))
        label = torch.from_numpy(label_cache[path].astype(np.float32))

        if self.augment:
            for ax in [1, 2, 3]:
                if random.random() < 0.5:
                    image = torch.flip(image, [ax])
                    label = torch.flip(label, [ax])
            image = image * torch.empty(4, 1, 1, 1).uniform_(0.9, 1.1)
            image = image + torch.empty(4, 1, 1, 1).uniform_(-0.1, 0.1)

        return image, label

In [7]:
train_ids, temp_ids = train_test_split(all_cases, test_size=0.3, random_state=SEED)
val_ids,   test_ids = train_test_split(temp_ids,  test_size=0.5, random_state=SEED)

print("Train:", len(train_ids), " Val:", len(val_ids), " Test:", len(test_ids))

train_loader = DataLoader(
    BraTSDataset(train_ids, augment=True),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, drop_last=True
)
val_loader = DataLoader(
    BraTSDataset(val_ids, augment=False),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True
)
test_loader = DataLoader(
    BraTSDataset(test_ids, augment=False),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True
)

print("Train batches:", len(train_loader))
print("Val batches:  ", len(val_loader))

Train: 257  Val: 55  Test: 56
Train batches: 128
Val batches:   28


## Swin-UNETR Architecture
- **Patch partition**: 2×2×2 voxel tokens  
- **4 Swin stages**: each with window-based self-attention + patch merging (embed doubles per stage: 48→96→192→384)
- **Shifted windows**: cyclic shift + masking for boundary attention in 3-D  
- **Decoder**: symmetric CNN upsample with skip connections from every stage

In [8]:
# ---------------------------------------------------------------------------
# Swin-UNETR -- pure PyTorch, no monai/timm required
# ---------------------------------------------------------------------------

def window_partition_3d(x, window_size):
    B, D, H, W, C = x.shape
    Wd = Wh = Ww = window_size
    x = x.view(B, D // Wd, Wd, H // Wh, Wh, W // Ww, Ww, C)
    windows = x.permute(0, 1, 3, 5, 2, 4, 6, 7).contiguous()
    windows = windows.view(-1, Wd, Wh, Ww, C)
    return windows


def window_reverse_3d(windows, window_size, D, H, W):
    Wd = Wh = Ww = window_size
    B = int(windows.shape[0] / ((D // Wd) * (H // Wh) * (W // Ww)))
    x = windows.view(B, D // Wd, H // Wh, W // Ww, Wd, Wh, Ww, -1)
    x = x.permute(0, 1, 4, 2, 5, 3, 6, 7).contiguous()
    x = x.view(B, D, H, W, -1)
    return x


def _pad_to_window(x, window_size):
    """Pad (B, D, H, W, C) so each spatial dim is divisible by window_size."""
    B, D, H, W, C = x.shape
    pad_d = (window_size - D % window_size) % window_size
    pad_h = (window_size - H % window_size) % window_size
    pad_w = (window_size - W % window_size) % window_size
    if pad_d or pad_h or pad_w:
        x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h, 0, pad_d))
    return x, (pad_d, pad_h, pad_w)


def _unpad(x, pad_d, pad_h, pad_w):
    """Remove padding added by _pad_to_window."""
    D, H, W = x.shape[1], x.shape[2], x.shape[3]
    return x[
        :,
        0 : D - pad_d if pad_d else D,
        0 : H - pad_h if pad_h else H,
        0 : W - pad_w if pad_w else W,
        :,
    ]


class WindowAttention3D(nn.Module):
    """3-D window-based multi-head self-attention with relative position bias."""
    def __init__(self, dim, window_size, num_heads, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.dim         = dim
        self.window_size = window_size
        self.num_heads   = num_heads
        head_dim         = dim // num_heads
        self.scale       = head_dim ** -0.5

        ws = window_size
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2*ws-1)**3, num_heads)
        )
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

        coords = torch.stack(
            torch.meshgrid(
                torch.arange(ws), torch.arange(ws), torch.arange(ws),
                indexing='ij'
            )
        )  # (3, ws, ws, ws)
        coords_flat = coords.flatten(1)
        rel = coords_flat[:, :, None] - coords_flat[:, None, :]
        rel = rel.permute(1, 2, 0).contiguous()
        rel[:, :, 0] += ws - 1
        rel[:, :, 1] += ws - 1
        rel[:, :, 2] += ws - 1
        rel[:, :, 0] *= (2*ws-1) ** 2
        rel[:, :, 1] *= (2*ws-1)
        rel_pos_idx = rel.sum(-1)
        self.register_buffer("relative_position_index", rel_pos_idx)

        self.qkv       = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj      = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self.softmax   = nn.Softmax(dim=-1)

    def forward(self, x, mask=None):
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)

        idx  = self.relative_position_index.view(-1)
        bias = self.relative_position_bias_table[idx].view(
            self.window_size**3, self.window_size**3, -1
        ).permute(2, 0, 1).contiguous().unsqueeze(0)
        attn = attn + bias

        if mask is not None:
            nW   = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N)
            attn = attn + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)

        attn = self.softmax(attn)
        attn = self.attn_drop(attn)
        x    = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x    = self.proj_drop(self.proj(x))
        return x


class SwinTransformerBlock3D(nn.Module):
    """Swin Transformer block with optional shifted window."""
    def __init__(self, dim, input_resolution, num_heads, window_size=7,
                 shift_size=0, mlp_ratio=4., drop=0., attn_drop=0.):
        super().__init__()
        self.dim              = dim
        self.input_resolution = input_resolution  # (D, H, W) in tokens
        self.num_heads        = num_heads
        self.window_size      = window_size
        self.shift_size       = shift_size

        self.norm1 = nn.LayerNorm(dim)
        self.attn  = WindowAttention3D(
            dim, window_size, num_heads, attn_drop=attn_drop, proj_drop=drop
        )
        self.norm2 = nn.LayerNorm(dim)
        hidden_dim = int(dim * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, hidden_dim), nn.GELU(), nn.Dropout(drop),
            nn.Linear(hidden_dim, dim), nn.Dropout(drop)
        )

        # Cyclic-shift attention mask built with padded resolution
        if shift_size > 0:
            D, H, W = input_resolution
            Dp = int(math.ceil(D / window_size)) * window_size
            Hp = int(math.ceil(H / window_size)) * window_size
            Wp = int(math.ceil(W / window_size)) * window_size
            img_mask = torch.zeros(1, Dp, Hp, Wp, 1)
            slices = [slice(0, -window_size),
                      slice(-window_size, -shift_size),
                      slice(-shift_size, None)]
            cnt = 0
            for d in slices:
                for h in slices:
                    for w in slices:
                        img_mask[:, d, h, w, :] = cnt
                        cnt += 1
            mask_windows = window_partition_3d(img_mask, window_size)
            mask_windows = mask_windows.view(-1, window_size**3)
            attn_mask    = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask    = attn_mask.masked_fill(attn_mask != 0, -100.0)
            attn_mask    = attn_mask.masked_fill(attn_mask == 0, 0.0)
        else:
            attn_mask = None

        self.register_buffer("attn_mask", attn_mask)

    def forward(self, x):
        D, H, W = self.input_resolution
        B, L, C = x.shape
        shortcut = x
        x = self.norm1(x).view(B, D, H, W, C)

        # Pad so spatial dims are divisible by window_size
        x, (pad_d, pad_h, pad_w) = _pad_to_window(x, self.window_size)
        _, Dp, Hp, Wp, _ = x.shape

        if self.shift_size > 0:
            shifted = torch.roll(x, shifts=(-self.shift_size,) * 3, dims=(1, 2, 3))
        else:
            shifted = x

        windows = window_partition_3d(shifted, self.window_size)
        windows = windows.view(-1, self.window_size**3, C)

        # Rebuild mask if padded size differs from the stored one
        if self.shift_size > 0:
            n_windows = (Dp // self.window_size) * (Hp // self.window_size) * (Wp // self.window_size)
            if self.attn_mask is None or self.attn_mask.shape[0] != n_windows:
                img_mask = torch.zeros(1, Dp, Hp, Wp, 1, device=x.device)
                slices = [slice(0, -self.window_size),
                          slice(-self.window_size, -self.shift_size),
                          slice(-self.shift_size, None)]
                cnt = 0
                for d in slices:
                    for h in slices:
                        for w in slices:
                            img_mask[:, d, h, w, :] = cnt
                            cnt += 1
                mask_windows = window_partition_3d(img_mask, self.window_size)
                mask_windows = mask_windows.view(-1, self.window_size**3)
                attn_mask    = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
                attn_mask    = attn_mask.masked_fill(attn_mask != 0, -100.0)
                attn_mask    = attn_mask.masked_fill(attn_mask == 0, 0.0)
            else:
                attn_mask = self.attn_mask
        else:
            attn_mask = None

        attn_out = self.attn(windows, mask=attn_mask)
        attn_out = attn_out.view(-1, self.window_size, self.window_size, self.window_size, C)
        shifted  = window_reverse_3d(attn_out, self.window_size, Dp, Hp, Wp)

        if self.shift_size > 0:
            x = torch.roll(shifted, shifts=(self.shift_size,) * 3, dims=(1, 2, 3))
        else:
            x = shifted

        # Remove padding
        x = _unpad(x, pad_d, pad_h, pad_w)

        x = x.reshape(B, L, C)
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x


class PatchMerging3D(nn.Module):
    """Merge 2x2x2 patches and double channels (downscale by 2x in each axis)."""
    def __init__(self, input_resolution, dim):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim    = dim
        self.norm   = nn.LayerNorm(8 * dim)
        self.reduce = nn.Linear(8 * dim, 2 * dim, bias=False)

    def forward(self, x):
        D, H, W = self.input_resolution
        B, L, C = x.shape
        x = x.view(B, D, H, W, C)
        patches = []
        for i in range(2):
            for j in range(2):
                for k in range(2):
                    patches.append(x[:, i::2, j::2, k::2, :])
        x = torch.cat(patches, dim=-1)           # (B, D/2, H/2, W/2, 8C)
        x = x.view(B, -1, 8 * C)
        x = self.norm(x)
        x = self.reduce(x)
        return x


class SwinStage3D(nn.Module):
    """One stage of the Swin encoder: alternating normal + shifted blocks."""
    def __init__(self, dim, input_resolution, depth, num_heads,
                 window_size=7, mlp_ratio=4., drop=0., attn_drop=0.,
                 downsample=True):
        super().__init__()
        self.blocks = nn.ModuleList([
            SwinTransformerBlock3D(
                dim=dim,
                input_resolution=input_resolution,
                num_heads=num_heads,
                window_size=window_size,
                shift_size=0 if (i % 2 == 0) else window_size // 2,
                mlp_ratio=mlp_ratio,
                drop=drop, attn_drop=attn_drop,
            )
            for i in range(depth)
        ])
        if downsample:
            self.downsample = PatchMerging3D(input_resolution, dim)
        else:
            self.downsample = None

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        skip = x
        if self.downsample is not None:
            x = self.downsample(x)
        return x, skip


class PatchEmbed3D(nn.Module):
    """Split volume into non-overlapping patches and project."""
    def __init__(self, in_ch=4, patch_size=2, embed_dim=48):
        super().__init__()
        self.proj = nn.Conv3d(in_ch, embed_dim,
                              kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)                      # (B, E, D', H', W')
        B, E, D, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)      # (B, D'H'W', E)
        x = self.norm(x)
        return x, (D, H, W)


class DoubleConv3D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.InstanceNorm3d(out_ch, affine=True),
            nn.LeakyReLU(0.01, inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SwinUNETR(nn.Module):
    """
    Swin-UNETR: Swin Transformers for Semantic Segmentation of Brain Tumors
    Hatamizadeh et al., MICCAI 2022.

    Encoder: 4 Swin stages with patch_size=2, window_size=7
      Stage 0: 64x64x64 tokens, dim=48
      Stage 1: 32x32x32 tokens, dim=96
      Stage 2: 16x16x16 tokens, dim=192
      Stage 3: 8x8x8   tokens, dim=384
    Decoder: symmetric CNN upsampling
    """
    def __init__(self, in_ch=4, num_classes=3,
                 img_size=128, patch_size=2,
                 embed_dim=48, window_size=7,
                 depths=(2, 2, 2, 2),
                 num_heads=(3, 6, 12, 24)):
        super().__init__()

        t0 = img_size // patch_size              # 64
        res = [
            (t0,     t0,     t0),
            (t0//2,  t0//2,  t0//2),
            (t0//4,  t0//4,  t0//4),
            (t0//8,  t0//8,  t0//8),
        ]
        dims = [embed_dim * (2**i) for i in range(4)]   # 48, 96, 192, 384

        self.patch_embed = PatchEmbed3D(in_ch, patch_size, embed_dim)

        self.stage0 = SwinStage3D(dims[0], res[0], depths[0], num_heads[0],
                                   window_size, downsample=True)
        self.stage1 = SwinStage3D(dims[1], res[1], depths[1], num_heads[1],
                                   window_size, downsample=True)
        self.stage2 = SwinStage3D(dims[2], res[2], depths[2], num_heads[2],
                                   window_size, downsample=True)
        self.stage3 = SwinStage3D(dims[3], res[3], depths[3], num_heads[3],
                                   window_size, downsample=False)

        self.proj0 = nn.Linear(dims[0], dims[0])
        self.proj1 = nn.Linear(dims[1], dims[1])
        self.proj2 = nn.Linear(dims[2], dims[2])
        self.proj3 = nn.Linear(dims[3], dims[3])

        self.norm0 = nn.LayerNorm(dims[0])
        self.norm1 = nn.LayerNorm(dims[1])
        self.norm2 = nn.LayerNorm(dims[2])
        self.norm3 = nn.LayerNorm(dims[3])

        self.res = res

        # ---- Decoder ----
        self.up3 = nn.ConvTranspose3d(dims[3], dims[2], 2, 2)
        self.dc3 = DoubleConv3D(dims[2] + dims[2], dims[2])

        self.up2 = nn.ConvTranspose3d(dims[2], dims[1], 2, 2)
        self.dc2 = DoubleConv3D(dims[1] + dims[1], dims[1])

        self.up1 = nn.ConvTranspose3d(dims[1], dims[0], 2, 2)
        self.dc1 = DoubleConv3D(dims[0] + dims[0], dims[0])

        self.up0 = nn.ConvTranspose3d(dims[0], 32, 2, 2)
        self.dc0 = DoubleConv3D(32, 32)

        self.out = nn.Conv3d(32, num_classes, 1)

    def _seq_to_vol(self, tokens, res):
        B, N, C = tokens.shape
        D, H, W = res
        return tokens.transpose(1, 2).reshape(B, C, D, H, W)

    def forward(self, x):
        tokens, (D0, H0, W0) = self.patch_embed(x)

        t1, skip0 = self.stage0(tokens)
        t2, skip1 = self.stage1(t1)
        t3, skip2 = self.stage2(t2)
        t4, skip3 = self.stage3(t3)

        s0 = self._seq_to_vol(self.norm0(skip0), self.res[0])
        s1 = self._seq_to_vol(self.norm1(skip1), self.res[1])
        s2 = self._seq_to_vol(self.norm2(skip2), self.res[2])
        s3 = self._seq_to_vol(self.norm3(skip3), self.res[3])

        d3 = self.dc3(torch.cat([self.up3(s3), s2], dim=1))
        d2 = self.dc2(torch.cat([self.up2(d3), s1], dim=1))
        d1 = self.dc1(torch.cat([self.up1(d2), s0], dim=1))
        d0 = self.dc0(self.up0(d1))                      # -> 128^3

        return self.out(d0)


In [9]:
def region_loss(logits, target):
    pred  = torch.sigmoid(logits)
    inter = (pred * target).sum(dim=(0, 2, 3, 4))
    union = pred.sum(dim=(0, 2, 3, 4)) + target.sum(dim=(0, 2, 3, 4))
    dice  = 1 - (2 * inter + 1e-6) / (union + 1e-6)
    bce   = F.binary_cross_entropy_with_logits(logits, target)
    return dice.mean() + bce

In [10]:
model = SwinUNETR(
    in_ch=4, num_classes=3,
    img_size=IMG_SIZE,
    patch_size=PATCH_SIZE,
    embed_dim=EMBED_DIM,
    window_size=WINDOW_SIZE,
    depths=DEPTHS,
    num_heads=NUM_HEADS,
)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model = model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.cuda.amp.GradScaler()  # FIX: deprecated torch.amp.GradScaler removed

print("Parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


Parameters: 10,658,581


/tmp/ipykernel_24/3826494808.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler()  # FIX: deprecated torch.amp.GradScaler removed


In [11]:
def dice_score(pred_bin, target_bin):
    inter = (pred_bin & target_bin).sum().float()
    return ((2 * inter + 1e-6) / (pred_bin.sum() + target_bin.sum() + 1e-6)).item()

def brats_metrics(pred_logits, target):
    pred = torch.sigmoid(pred_logits) > 0.5
    wt   = dice_score(pred[:, 0], target[:, 0].bool())
    tc   = dice_score(pred[:, 1], target[:, 1].bool())
    et   = dice_score(pred[:, 2], target[:, 2].bool())
    return wt, tc, et

In [12]:
best_mean_dice   = 0.0
patience_counter = 0
train_start      = time.time()

for epoch in range(EPOCHS):
    t0         = time.time()
    current_lr = scheduler.get_last_lr()[0]
    elapsed    = (t0 - train_start) / 60
    print(f"\n{'='*60}")
    print(f"Epoch {epoch+1}/{EPOCHS}   LR: {current_lr:.6f}   Elapsed: {elapsed:.1f}min")

    model.train()
    train_loss = 0.0
    for x, y in tqdm(train_loader, desc="Train", leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda"):
            out  = model(x)
            loss = region_loss(out, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    scheduler.step()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0.0
    wt_scores, tc_scores, et_scores = [], [], []

    with torch.no_grad():
        for x, y in tqdm(val_loader, desc="Val", leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.amp.autocast("cuda"):
                out  = model(x)
                loss = region_loss(out, y)
            val_loss += loss.item()
            wt, tc, et = brats_metrics(out, y)
            wt_scores.append(wt)
            tc_scores.append(tc)
            et_scores.append(et)

    val_loss  /= len(val_loader)
    wt_mean    = np.mean(wt_scores)
    tc_mean    = np.mean(tc_scores)
    et_mean    = np.mean(et_scores)
    mean_dice  = (wt_mean + tc_mean + et_mean) / 3
    epoch_time = time.time() - t0

    print(f"Train Loss: {train_loss:.4f}   Val Loss: {val_loss:.4f}")
    print(f"WT: {wt_mean:.4f}   TC: {tc_mean:.4f}   ET: {et_mean:.4f}   Mean: {mean_dice:.4f}")
    print(f"Epoch time: {epoch_time:.1f}s   Total elapsed: {(time.time()-train_start)/60:.1f}min")

    if mean_dice > best_mean_dice:
        best_mean_dice   = mean_dice
        patience_counter = 0
        torch.save(model.state_dict(), "best_model_swinunetr.pth")
        print(f"Best model saved  (mean dice: {best_mean_dice:.4f})")
    else:
        patience_counter += 1
        print(f"No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

print(f"\nTraining complete. Total time: {(time.time()-train_start)/60:.1f}min")


Epoch 1/10   LR: 0.000100   Elapsed: 0.0min


Train:   0%|          | 0/128 [00:00<?, ?it/s]

Val:   0%|          | 0/28 [00:00<?, ?it/s]

Train Loss: 1.5204   Val Loss: 1.4600
WT: 0.5982   TC: 0.4328   ET: 0.2473   Mean: 0.4261
Epoch time: 296.5s   Total elapsed: 4.9min
Best model saved  (mean dice: 0.4261)

Epoch 2/10   LR: 0.000098   Elapsed: 5.0min


Train:   0%|          | 0/128 [00:00<?, ?it/s]

...

Val:   0%|          | 0/28 [00:00<?, ?it/s]

Train Loss: 1.1983   Val Loss: 1.1977
WT: 0.8066   TC: 0.6630   ET: 0.4655   Mean: 0.6451
Epoch time: 295.6s   Total elapsed: 44.6min
No improvement (1/6)

Epoch 10/10   LR: 0.000002   Elapsed: 44.6min


Train:   0%|          | 0/128 [00:00<?, ?it/s]

Val:   0%|          | 0/28 [00:00<?, ?it/s]

Train Loss: 1.1954   Val Loss: 1.1964
WT: 0.8213   TC: 0.6817   ET: 0.4835   Mean: 0.6622
Epoch time: 295.6s   Total elapsed: 49.5min
No improvement (2/6)

Training complete. Total time: 49.5min


In [13]:
# FIX: map_location + strip DataParallel 'module.' prefix if present
state = torch.load("best_model_swinunetr.pth", map_location=DEVICE)
if any(k.startswith("module.") for k in state.keys()):
    state = {k[len("module."):]: v for k, v in state.items()}
model.load_state_dict(state)
model.eval()

test_loss = 0.0
wt_scores, tc_scores, et_scores = [], [], []

with torch.no_grad():
    for x, y in tqdm(test_loader, desc="Test"):
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.amp.autocast("cuda"):
            out  = model(x)
            loss = region_loss(out, y)
        test_loss += loss.item()
        wt, tc, et = brats_metrics(out, y)
        wt_scores.append(wt)
        tc_scores.append(tc)
        et_scores.append(et)

test_loss /= len(test_loader)
wt_mean    = np.mean(wt_scores)
tc_mean    = np.mean(tc_scores)
et_mean    = np.mean(et_scores)
mean_dice  = (wt_mean + tc_mean + et_mean) / 3

print(f"\n{'='*60}")
print("FINAL TEST RESULTS  --  Swin-UNETR")
print(f"Test Loss : {test_loss:.4f}")
print(f"WT Dice   : {wt_mean:.4f}")
print(f"TC Dice   : {tc_mean:.4f}")
print(f"ET Dice   : {et_mean:.4f}")
print(f"Mean Dice : {mean_dice:.4f}")


RuntimeError: Error(s) in loading state_dict for DataParallel:
	Missing key(s) in state_dict: "module.patch_embed.proj.weight", "module.patch_embed.proj.bias", "module.patch_embed.norm.weight", "module.patch_embed.norm.bias", "module.stage0.blocks.0.norm1.weight", "module.stage0.blocks.0.norm1.bias", "module.stage0.blocks.0.attn.relative_position_bias_table", "module.stage0.blocks.0.attn.relative_position_index", "module.stage0.blocks.0.attn.qkv.weight", "module.stage0.blocks.0.attn.qkv.bias", "module.stage0.blocks.0.attn.proj.weight", "module.stage0.blocks.0.attn.proj.bias", "module.stage0.blocks.0.norm2.weight", "module.stage0.blocks.0.norm2.bias", "module.stage0.blocks.0.mlp.0.weight", "module.stage0.blocks.0.mlp.0.bias", "module.stage0.blocks.0.mlp.3.weight", "module.stage0.blocks.0.mlp.3.bias", "module.stage0.blocks.1.attn_mask", "module.stage0.blocks.1.norm1.weight", "module.stage0.blocks.1.norm1.bias", "module.stage0.blocks.1.attn.relative_position_bias_table", "module.stage0.bl